# 5 · The Kalman filter

**Recap** ([theory](https://github.com/Rudip1/learn-state-estimation/blob/main/1_theory/05_kalman_filter.md)).
For $x_k = Fx_{k-1} + w_k$, $z_k = Hx_k + v_k$ with Gaussian noises the belief stays Gaussian. Predict:
$\bar x = F\hat x$, $\bar P = FPF^T + Q$. Correct: innovation $\nu = z - H\bar x$, $S = H\bar PH^T + R$,
$K = \bar PH^TS^{-1}$, $\hat x = \bar x + K\nu$, $P = (I-KH)\bar P(I-KH)^T + KRK^T$ (Joseph form, valid for any $K$).
A filter is consistent when NEES $(x-\hat x)^TP^{-1}(x-\hat x) \sim \chi^2_n$ and NIS $\nu^TS^{-1}\nu \sim \chi^2_m$;
averaged over $N$ runs they must stay inside $[\chi^2_{Nn}(\alpha/2), \chi^2_{Nn}(1-\alpha/2)]/N$. With constant
matrices the gain converges to a steady state set by the ratio $Q/R$.

In [ ]:
# Installs the module when it is not available yet (e.g. on Colab). Building the C++ takes a few minutes.
try:
    import state_estimation
except ImportError:
    %pip install -q git+https://github.com/Rudip1/learn-state-estimation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import state_estimation as se
from state_estimation import plotting

plt.rcParams.update({"figure.dpi": 90, "figure.figsize": (6, 4.5)})

## Tracking a target with constant velocity

A point moves along a line with white-noise acceleration ($q = 0.2$); we measure its position with
$\sigma = 0.5$ m every 0.1 s and estimate position **and velocity**.

In [ ]:
dt, q, steps = 0.1, 0.2, 200
model = se.constant_velocity_model(1, dt, q)
H = np.array([[1.0, 0.0]])
R = np.array([[0.25]])
x0, P0 = np.array([0.0, 1.0]), np.diag([1.0, 1.0])
run = se.simulate_linear_system(model.F, model.Q, H, R, x0, steps, se.Rng(0))

def kalman_run(run, Q=model.Q, R=R, x0=x0, P0=P0):
    kf = se.KalmanFilter(x0, P0)
    xs, Ps, nis = [x0], [P0], []
    for z in run.measurements:
        kf.predict(model.F, Q)
        nis.append(kf.update(z, H, R).nis)
        xs.append(kf.x.copy()); Ps.append(kf.P.copy())
    return np.array(xs), np.array(Ps), np.array(nis)

xs, Ps, _ = kalman_run(run)
t = np.arange(steps + 1) * dt
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for i, (ax, name) in enumerate(zip(axes, ["position [m]", "velocity [m/s]"])):
    sd = np.sqrt(Ps[:, i, i])
    ax.plot(t, run.states[:, i], "k", label="truth")
    if i == 0:
        ax.plot(t[1:], run.measurements[:, 0], ".", ms=3, color="C2", label="measurements")
    ax.plot(t, xs[:, i], "C0", label="estimate")
    ax.fill_between(t, xs[:, i] - 2 * sd, xs[:, i] + 2 * sd, color="C0", alpha=0.2, label="±2σ")
    ax.set_xlabel("t [s]"); ax.set_ylabel(name); ax.legend(fontsize=8)
plt.show()

### ✏️ Exercise 1 — the update step

Implement the correction (5.7)–(5.10) with the Joseph-form covariance (5.12). Use `np.linalg.solve` rather
than an explicit inverse for the gain.

In [ ]:
def kf_update(x, P, z, H, R):
    # ✏️ your code here
    raise NotImplementedError("exercise")


In [ ]:
rng = np.random.default_rng(3)
for _ in range(20):
    A = rng.normal(size=(4, 4)); P = A @ A.T + np.eye(4)
    Hh = rng.normal(size=(2, 4)); Rr = np.diag(rng.uniform(0.1, 1.0, 2))
    x, z = rng.normal(size=4), rng.normal(size=2)
    kf = se.KalmanFilter(x, P); kf.update(z, Hh, Rr)
    xn, Pn = kf_update(x, P, z, Hh, Rr)
    assert np.allclose(xn, kf.x) and np.allclose(Pn, kf.P)
print("✔ matches the C++ filter")

### ✏️ Exercise 2 — the steady-state gain

For the random walk $x_k = x_{k-1} + w_k$, $z_k = x_k + v_k$ (variances $q$, $r$), implement `steady_gain(q, r)` from
(5.17). Check it against `se.steady_state_covariance` and plot $K_\infty$ against $q/r$: the filter is tuned by
a single number.

In [ ]:
def steady_gain(q, r):
    # ✏️ your code here
    raise NotImplementedError("exercise")


In [ ]:
I1 = np.eye(1)
for q_, r_ in [(1, 4), (0.01, 1), (10, 0.1)]:
    pbar = se.steady_state_covariance(I1, I1, I1 * q_, I1 * r_)[0, 0]
    assert np.isclose(steady_gain(q_, r_), pbar / (pbar + r_))
print(f"q = 1, r = 4: K∞ = {steady_gain(1, 4):.4f}")
ratios = np.logspace(-4, 4, 100)
plt.semilogx(ratios, [steady_gain(rr, 1.0) for rr in ratios])
plt.xlabel("q / r"); plt.ylabel("steady-state gain K∞"); plt.grid(alpha=0.3); plt.show()

### ✏️ Exercise 3 — a Monte Carlo consistency test

Write `average_nees(Q_filter, runs=50)`: simulate `runs` trajectories with the true model, filter each one with
process noise `Q_filter`, and return the NEES averaged over the runs at each step. With the true $Q$ about 95 % of
the steps must lie inside `se.average_chi2_bounds(2, runs)`.

In [ ]:
def average_nees(Q_filter, runs=50, seed=1):
    # ✏️ your code here
    raise NotImplementedError("exercise")


In [ ]:
lo, hi = se.average_chi2_bounds(2, 50)
nees_ok = average_nees(model.Q)
inside = np.mean((nees_ok >= lo) & (nees_ok <= hi))
print(f"bounds [{lo:.3f}, {hi:.3f}], {100 * inside:.0f} % of the steps inside")
assert inside > 0.85

## 🔨 Break it — a mistuned process noise

Filter the same runs with $Q$ a hundred times too small (the filter believes the target moves very smoothly) and a
hundred times too large. Too small: NEES far above the bounds — **over-confident**; too large: below — pessimistic,
and noisier estimates than necessary. NIS shows the same without needing the truth.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for scale, color in ((0.01, "C3"), (1.0, "C0"), (100.0, "C2")):
    ax.semilogy(t[1:], average_nees(model.Q * scale), color=color, label=f"Q × {scale:g}")
ax.axhline(lo, color="k", ls="--"); ax.axhline(hi, color="k", ls="--")
ax.set_xlabel("t [s]"); ax.set_ylabel("average NEES (50 runs)"); ax.legend(); plt.show()

## 🔨 Break it — "the model is exact", Q = 0

With $Q = 0$ the covariance shrinks forever and the gain goes to zero: the filter stops listening. When the target
changes its velocity at $t = 10$ s, the estimate keeps going in the old direction, and the NIS explodes — the
measurements are screaming that something is wrong.

In [ ]:
truth = np.zeros((steps + 1, 2)); truth[0] = [0.0, 1.0]
for k in range(steps):
    v = 1.0 if k * dt < 10 else -0.5
    truth[k + 1] = [truth[k, 0] + v * dt, v]
noise_rng = se.Rng(5)
zs = truth[1:, :1] + 0.5 * np.array([noise_rng.normal() for _ in range(steps)])[:, None]
manoeuvre = type("Run", (), {"measurements": zs})()
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for Qf, name in ((np.zeros((2, 2)), "Q = 0"), (model.Q, "Q from the model")):
    est, P, nis = kalman_run(manoeuvre, Q=Qf)
    axes[0].plot(t, est[:, 0], label=name)
    axes[1].semilogy(t[1:], nis, label=name)
axes[0].plot(t, truth[:, 0], "k", label="truth"); axes[0].set_ylabel("position"); axes[0].legend()
axes[1].axhline(se.chi2_quantile(0.95, 1), color="k", ls="--", label="95 % gate")
axes[1].set_ylabel("NIS"); axes[1].legend(); plt.show()

## 🔨 Break it — the short covariance form with a suboptimal gain

Use a fixed, deliberately small gain. The Joseph form reports the covariance the errors actually have; the short
form $(I-KH)\bar P$ reports something else — here three times too large in position; with other gains it can be too
small, or not even positive definite.

In [ ]:
K_fixed = np.array([[0.05], [0.01]])
err, rep_j, rep_s = [], None, None
rng = se.Rng(8)
for _ in range(3000):
    start = se.sample_gaussian(x0, P0, 1, rng)[0]
    r = se.simulate_linear_system(model.F, model.Q, H, R, start, 50, rng)
    kj, ks = se.KalmanFilter(x0, P0), se.KalmanFilter(x0, P0)
    for z in r.measurements:
        for kf_, jos in ((kj, True), (ks, False)):
            kf_.predict(model.F, model.Q); kf_.update_with_gain(z, H, R, K_fixed, jos)
    err.append(r.states[-1] - kj.x)
    rep_j, rep_s = kj.P, ks.P
print("actual error covariance\n", np.cov(np.array(err).T).round(3))
print("Joseph form reports\n", rep_j.round(3))
print("short form reports\n", rep_s.round(3))

## What to remember

- Prediction adds $Q$, correction subtracts information; the gain balances them and depends on $Q/R$ only.
- The innovation is the filter's own health monitor: white, zero-mean, covariance $S$.
- Check consistency with NEES (simulation) and NIS (real data) against $\chi^2$ bounds over many runs.
- Use the Joseph form; never set $Q = 0$ for a system that can surprise you.